# PPITSAS: mean inference for time series across space

One run: data and chronological fitting -> target -> inference -> output.
Run from `examples/simulation/`, as with `ppi.ipynb`.

This stationary simulation has temporally and spatially correlated covariates
and errors. A shared linear predictor omits a quadratic term. Exact site-level
long-run means are known. This is an OpenPPI simulation case, not a
reproduction of the paper's weather experiment.

Chronology: **old unlabeled history -> recent rectification -> latest fitting**.
Pool sites only for fitting; do not shuffle blocks or flatten sites for HAC.
The paper neglects cross-block/training dependence asymptotically and gives a
proof sketch rather than a completed general coverage theorem.


In [1]:
import pandas as pd
import numpy as np

from openppi import Mean, get_method
from dataset.time_series_sim import TimeSeriesDGP
from time_series import prepare_records, run_simulation

In [2]:
# 1. Data and fitting: 1600 historical, 800 rectification, 200 fitting times.
dgp = TimeSeriesDGP(sites=9, phi=.7, error_phi=.5)
data = dgp.draw(2600, np.random.default_rng(20261029))
# Fit one shared linear model only on the latest 200 times across sites.
# Return one chronological PPIData per site.
records = prepare_records(data, unlabeled_size=1600,
                          rectification_size=800, train_size=200)

In [3]:
# 2. Select the mean target; PPITSAS does not support regression.
target = Mean(outcome_type='continuous')

In [4]:
# 3. Same corrected point estimate as PPI; separate HAC mean variances.
methods = {name: get_method(name).inference for name in ('PPI', 'PPITSAS')}
results = {name: [method(record, target=target) for record in records]
           for name, method in methods.items()}

In [5]:
# 4. Pointwise intervals at each location, not simultaneous bands.
rows = []
for name, site_results in results.items():
    for site, result in enumerate(site_results):
        low, high = result.confidence_interval
        rows.append(dict(site=site, method=name, truth=data['truth'][site],
                         estimate=result.estimate, low=low, high=high,
                         covered=low <= data['truth'][site] <= high))
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda x: f'{x:.4f}'))
print()
print('PPITSAS site 0 diagnostics:', results['PPITSAS'][0].diagnostics)

 site  method  truth  estimate    low   high  covered
    0     PPI 1.0000    1.0921 1.0294 1.1548    False
    1     PPI 1.0625    1.1509 1.0823 1.2194    False
    2     PPI 1.1250    1.1825 1.1183 1.2467     True
    3     PPI 1.1875    1.1572 1.0944 1.2200     True
    4     PPI 1.2500    1.2182 1.1544 1.2820     True
    5     PPI 1.3125    1.3363 1.2698 1.4029     True
    6     PPI 1.3750    1.4860 1.4175 1.5544    False
    7     PPI 1.4375    1.4969 1.4279 1.5659     True
    8     PPI 1.5000    1.5025 1.4321 1.5728     True
    0 PPITSAS 1.0000    1.0921 0.9820 1.2022     True
    1 PPITSAS 1.0625    1.1509 1.0252 1.2765     True
    2 PPITSAS 1.1250    1.1825 1.0743 1.2907     True
    3 PPITSAS 1.1875    1.1572 1.0491 1.2652     True
    4 PPITSAS 1.2500    1.2182 1.1041 1.3323     True
    5 PPITSAS 1.3125    1.3363 1.2250 1.4476     True
    6 PPITSAS 1.3750    1.4860 1.3690 1.6029     True
    7 PPITSAS 1.4375    1.4969 1.3730 1.6207     True
    8 PPITSAS 1.5000    1.50

## Repeated time-series simulation

Compare iid PPI and PPITSAS on the same records over 200 independent
replicates. Since sites within each replicate are correlated, coverage Monte
Carlo SE uses replicate-level average coverage. Finite-sample coverage depends
on the DGP, sample sizes and bandwidth; it is not a universal guarantee.

Command line: `python time_series.py --reps 200 --out results_time_series`.


In [6]:
per_rep_site, summary = run_simulation(reps=200)
print(summary.to_string(index=False))

 method  reps  coverage  avg_width  coverage_mc_se     rmse
    PPI   200  0.693333   0.126362        0.013185 0.062574
PPITSAS   200  0.928889   0.230602        0.007917 0.062574
